# Data Extraction and Web Scraping

We will use a real-world example to demonstrate the concepts of data extraction via REST API and GraphQL. The APIs we will be using are the [GitHub REST API](https://docs.github.com/en/rest) and [GitHub GraphQL API](https://docs.github.com/en/graphql).

They are both APIs provided by GitHub to access data on their platform. The GraphQL API is a newer API. You can access data such as user profiles, repositories, issues, pull requests, releases etc. from GitHub.

We will use the following libraries for this lesson:

- `requests` for making HTTP requests
- `json` for parsing JSON responses
- `pandas` and `numpy` for data manipulation
- `sqlalchemy` for writing data to database (DuckDB)
- `beautifulsoup4` for web scraping

## REST API

Usually, before we can make a request to an API, we need to register for an API key. This is for the API provider to track usage, prevent abuse and to authenticate users. For more information on how to authenticate with GitHub API, refer to the [GitHub API docs](https://docs.github.com/en/rest/overview/resources-in-the-rest-api?apiVersion=2022-11-28#authentication). 

For this lesson, you will need to a personal access token (API key). If you do not have an API key, you can create a new one by folllowing the instructions in the official GitHub documentation [here](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens#creating-a-fine-grained-personal-access-token).

We will be retrieving data from some popular repositories on GitHub.

### Setting up Github API

After getting Github API key, please create a file called `.env` at the root folder of this repository. Alternatively, you can run the following code **ONCE** to create the env file.


In [1]:
import os

# Cell: Create the .env Template File
# -----------------------------------
template_content = """# .env file for GitHub Access
# 1. REPLACE the placeholder value <YOUR-GITHUB-ACCESS-TOKEN> below with the actual
#    GitHub access token. e.g., ghp_XXXXX
# 2. Save the file.

GITHUB_TOKEN='<YOUR-GITHUB-ACCESS-TOKEN>'
"""

# Check if env file already there
env_file_path = '../.env'
if os.path.exists(env_file_path):
    print(f"Warning: The file '{env_file_path}' already exists.")
    print("Please check the file and update the values if necessary.")
    print("Please open the '../.env' file and replace the '<YOUR-GITHUB-ACCESS-TOKEN>' placeholders with your actual token.")
    exit(1)
else:
    # Use mode 'w' to overwrite the file cleanly
    try:
        with open('../.env', 'w') as f:
            f.write(template_content)

        print("Template file created: ../.env")
        print("Please open the '../.env' file and replace the '<YOUR-GITHUB-ACCESS-TOKEN>' placeholders with your actual token.")
    except Exception as e:
        print(f"Error creating file: {e}")

Template file created: ../.env
Please open the '../.env' file and replace the '<YOUR-GITHUB-ACCESS-TOKEN>' placeholders with your actual token.



Next, open the `.env` file and replace your Github token with the placeholder:

```text
GITHUB_TOKEN='<YOUR-GITHUB-ACCESS-TOKEN>'
```

Save and close the file when done.

> Please note that when you sync your work to Github, this file will not be synced. Please keep a copy of `.env` safe elsewhere.

In [2]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
access_token = os.getenv('GITHUB_TOKEN')

Github API endpoints are in the format 
```
https://api.github.com/repos/{owner}/{repo}
```
and appended with the resource you want to access.

For example, to access the `issues` resource, the endpoint is
```
https://api.github.com/repos/{owner}/{repo}/issues
```

---

Here are some common endpoints:

| Resource | Endpoint |
| --- | --- |
| Issues | `/issues` |
| Pull Requests | `/pulls` |
| Commits | `/commits` |
| Contributors | `/contributors` |
| Languages | `/languages` |
| Releases | `/releases` |
| Tags | `/tags` |
| Branches | `/branches` |
| Forks | `/forks` |
| Stargazers | `/stargazers` |
| Subscribers | `/subscribers` |
| Subscription | `/subscription` |

Here are the HTTP verbs for the Github API:

| Verb | Description |
| --- | --- |
| GET | Used for retrieving resources. |
| POST | Used for creating resources. |
| PATCH | Used for updating resources with partial JSON data. For instance, an Issue resource has title and body attributes. A PATCH request may accept one or more of the attributes to update the resource. |
| PUT | Used for replacing resources or collections. For PUT requests with no body attribute, be sure to set the Content-Length header to zero. |
| DELETE | Used for deleting resources. |

We will only be using the GET verb for this project, 


`Pandas` is an open-source library, its code repository is hosted on GitHub. You can access the repository at [https://github.com/pandas-dev/pandas](https://github.com/pandas-dev/pandas).

Let's try to extract the past `releases` (library versions) from the the repository. The API endpoint is `https://api.github.com/repos/pandas-dev/pandas/releases`.

The access token needs to be passed in the `Authorization` header. The token is a personal access token (PAT) which you can generate on GitHub. The token is used to authenticate you as a user and to authorize you to access the repository.

It is also recommended to pass in the following headers:

- `Accept: application/vnd.github+json` - to specify the preferred format of the response
- `X-GitHub-Api-Version:2022-11-28` - to specify the version of the API to use

---

If we use a `GET` request to access the endpoint, we will get a list of releases in JSON format (default limit of _30 results_ per page). More info on the `releases` resource [here](https://docs.github.com/en/rest/releases/releases?apiVersion=2022-11-28).

In [3]:
response = requests.get("https://api.github.com/repos/pandas-dev/pandas/releases", 
                        headers={"Accept": "application/vnd.github+json", "Authorization": f"Bearer {access_token}"})

We can check if the response is successful by checking the status code of the response. If the status code is 200, then the response is successful.

In [4]:
response.status_code

200

> **Troubleshooting**:
> **What if we have status code in the range of 400?**
>
> refer to https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status for details.
> - Check your token in dotenv if it is in order
> - Print variable `access_token` to see if the token is same as your token in dotenv file
> - Restart your kernel may solved the issue.
> - Check if your token has expired
> - Setup a new token from Github

Use the `.json()` method to return the JSON format of the response object.

In [5]:
releases_page_1 = response.json()

In [6]:
type(releases_page_1)

list

In [7]:
len(releases_page_1)

30

In [8]:
releases_page_1[0]

{'url': 'https://api.github.com/repos/pandas-dev/pandas/releases/320615205',
 'assets_url': 'https://api.github.com/repos/pandas-dev/pandas/releases/320615205/assets',
 'upload_url': 'https://uploads.github.com/repos/pandas-dev/pandas/releases/320615205/assets{?name,label}',
 'html_url': 'https://github.com/pandas-dev/pandas/releases/tag/v3.0.3',
 'id': 320615205,
 'author': {'login': 'jorisvandenbossche',
  'id': 1020496,
  'node_id': 'MDQ6VXNlcjEwMjA0OTY=',
  'avatar_url': 'https://avatars.githubusercontent.com/u/1020496?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/jorisvandenbossche',
  'html_url': 'https://github.com/jorisvandenbossche',
  'followers_url': 'https://api.github.com/users/jorisvandenbossche/followers',
  'following_url': 'https://api.github.com/users/jorisvandenbossche/following{/other_user}',
  'gists_url': 'https://api.github.com/users/jorisvandenbossche/gists{/gist_id}',
  'starred_url': 'https://api.github.com/users/jorisvandenbossche/starred{

In [9]:
releases_page_1[0]['name']

'pandas 3.0.3'

In [10]:
releases_page_1[0]['published_at']

'2026-05-11T18:23:29Z'

The latest version of `pandas` is the one shown above. Hence, the list is sorted from most recent to oldest.

While the last release on the list is:

In [11]:
releases_page_1[-1]['name']

'Pandas 1.5.2'

In [12]:
releases_page_1[-1]['published_at']

'2022-11-22T07:07:41Z'

Remember the default results per page is `30`. You can increase the number of results per page to a maximum of `100` by using the `per_page` query parameter.

In [13]:
response = requests.get("https://api.github.com/repos/pandas-dev/pandas/releases?per_page=100", 
                        headers={"Accept": "application/vnd.github+json", "Authorization": f"Bearer {access_token}"})

In [14]:
releases_page_1_100 = response.json()

In [15]:
len(releases_page_1_100)

100

In [15]:
releases_page_1_100[-1]

{'url': 'https://api.github.com/repos/pandas-dev/pandas/releases/2776482',
 'assets_url': 'https://api.github.com/repos/pandas-dev/pandas/releases/2776482/assets',
 'upload_url': 'https://uploads.github.com/repos/pandas-dev/pandas/releases/2776482/assets{?name,label}',
 'html_url': 'https://github.com/pandas-dev/pandas/releases/tag/v0.18.0rc2',
 'id': 2776482,
 'author': {'login': 'jreback',
  'id': 953992,
  'node_id': 'MDQ6VXNlcjk1Mzk5Mg==',
  'avatar_url': 'https://avatars.githubusercontent.com/u/953992?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/jreback',
  'html_url': 'https://github.com/jreback',
  'followers_url': 'https://api.github.com/users/jreback/followers',
  'following_url': 'https://api.github.com/users/jreback/following{/other_user}',
  'gists_url': 'https://api.github.com/users/jreback/gists{/gist_id}',
  'starred_url': 'https://api.github.com/users/jreback/starred{/owner}{/repo}',
  'subscriptions_url': 'https://api.github.com/users/jreback/subsc

The default page is `1`, you can get the 2nd page by passing the `page` query parameter.

In [17]:
response = requests.get("https://api.github.com/repos/pandas-dev/pandas/releases?per_page=100&page=2", 
                        headers={"Accept": "application/vnd.github+json", "Authorization": f"Bearer {access_token}"})

In [16]:
releases_page_2_100 = response.json()

In [17]:
len(releases_page_2_100)

100


We're interested in the following fields and want to save them into a dataframe:

- `tag_name`: The version name/number of the release.
- `published_at`: The date and time when the release was published.
- `body`: The release notes.

In [18]:
import pandas as pd

In [19]:
releases_with_essential_fields = [{"version": release["tag_name"], "published_at": release["published_at"], "summary": release["body"]} for release in releases_page_1_100]

In [21]:
pandas_releases = pd.DataFrame(releases_with_essential_fields)

pandas_releases

,version,published_at,summary
0,v3.0.3,2026-05-11T18:23:29Z,We are pleased to announce the release of pand...
1,v3.0.2,2026-03-30T20:14:45Z,We are pleased to announce the release of pand...
2,v3.0.1,2026-02-17T21:56:03Z,We are pleased to announce the release of pand...
3,v3.0.0,2026-01-21T15:23:43Z,We are pleased to announce the release of pand...
4,v3.0.0rc2,2026-01-14T22:17:15Z,
...,...,...,...
95,v0.19.0,2016-10-02T14:17:50Z,This is a major release from 0.18.1 and includ...
96,v0.19.0rc1,2016-09-07T20:50:22Z,**RELEASE CANDIDATE**\n\nThis is a major relea...
97,v0.18.1,2016-05-03T14:48:48Z,This is a minor release from 0.18.0 and includ...
98,v0.18.0,2016-03-12T15:12:32Z,This is a major release from 0.17.1 and includ...


In [27]:
pandas_releases.dtypes

version                      object
published_at    datetime64[ns, UTC]
summary                      object
dtype: object

In [22]:
# convert `published_at` to the correct datetime format
pandas_releases['published_at'] = pd.to_datetime(pandas_releases['published_at'])

> What is the average interval (duration or differences in time) for pandas' releases (in no of days)?
>
> Hint 1: Sort the dataframe by `published_at` in ascending order.
> 
> Hint 2: Use [diff()](https://pandas.pydata.org/docs/reference/api/pandas.Series.diff.html) to calculate the time deltas between each `published_at`.
>
> Refer to [Time deltas](https://pandas.pydata.org/pandas-docs/stable/user_guide/timedeltas.html#time-deltas) for more info.

> Print the `summary` of the first (1.0.0) and second (2.0.0) major releases of pandas.

In [40]:
pandas_releases = pandas_releases.sort_values('published_at')

pandas_releases['time_since_last_relesae'] = pandas_releases['published_at'].diff()
pandas_releases.iloc[0:20]

,version,published_at,summary,time_since_last_relesae
99,v0.18.0rc2,2016-03-09 14:47:45+00:00,**RELEASE CANDIDATE 2**\n\nTHIS IS NOT A PRODU...,NaT
98,v0.18.0,2016-03-12 15:12:32+00:00,This is a major release from 0.17.1 and includ...,3 days 00:24:47
97,v0.18.1,2016-05-03 14:48:48+00:00,This is a minor release from 0.18.0 and includ...,51 days 23:36:16
96,v0.19.0rc1,2016-09-07 20:50:22+00:00,**RELEASE CANDIDATE**\n\nThis is a major relea...,127 days 06:01:34
95,v0.19.0,2016-10-02 14:17:50+00:00,This is a major release from 0.18.1 and includ...,24 days 17:27:28
94,v0.19.1,2016-11-03 15:29:59+00:00,This is a minor bug-fix release from 0.19.0 an...,32 days 01:12:09
93,v0.19.2,2016-12-24 16:47:17+00:00,This is a minor bug-fix release in the 0.19.x ...,51 days 01:17:18
92,v0.20.0rc1,2017-04-22 03:56:17+00:00,**RELEASE CANDIDATE 1**\r\n\r\nThis is a major...,118 days 11:09:00
91,v0.20.0rc2,2017-05-04 15:10:30+00:00,**RELEASE CANDIDATE 2**\r\n\r\nThis is a major...,12 days 11:14:13
90,v0.20.0,2017-05-05 02:37:00+00:00,This is a major release from 0.19.2 and includ...,0 days 11:26:30


Let's write the dataframe to a table in DuckDB.

First, we need to import the required libraries and get the absolute path of the project.

In [41]:
import sqlalchemy as sqla
import os 

parent_dir = os.path.abspath(os.path.pardir)

In [42]:
parent_dir

'/home/nainar/5m-data-2.4-data-extraction-scraping'

In [43]:
engine = sqla.create_engine(f'duckdb:///{parent_dir}/output/unit-2-3.db')

In [44]:
try:
    pandas_releases.to_sql("pandas_releases", engine)
    print("DataFrame successfully written to the database.")
except Exception as e:
    print(f"{e}")

DataFrame successfully written to the database.


/tmp/ipykernel_3349/2598590139.py:2: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  pandas_releases.to_sql("pandas_releases", engine)


In [45]:
# Safely dispose of the engine and close all connections
# You can only use dbeaver or dbgate to access this table after engine.dispose().
engine.dispose()

Let's use another resource (endpoint) now. There are many active contributors to the pandas library, and suppose we want to know how many commits each contributor has contributed. We can retrieve the data using the `stats/contributors` endpoint.

More info on the endpoint can be found [here](https://docs.github.com/en/rest/metrics/statistics?apiVersion=2022-11-28).

In [46]:
response = requests.get("https://api.github.com/repos/pandas-dev/pandas/stats/contributors", 
                        headers={"Accept": "application/vnd.github+json", "Authorization": f"Bearer {access_token}"})

In [47]:
response.status_code

200

In [48]:
contributors = response.json()

In [49]:
type(contributors)

list

In [51]:
len(contributors)

100

In [54]:
contributors[0]

{'total': 21,
 'weeks': [{'w': 1248566400, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1249171200, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1249776000, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1250380800, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1250985600, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1251590400, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1252195200, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1252800000, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1253404800, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1254009600, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1254614400, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1255219200, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1255824000, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1256428800, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1257033600, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1257638400, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1258243200, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1258848000, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1259452800, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1260057600, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1260662400, 'a': 0, 'd': 0, 'c': 0},
  {'w': 1261267200, 'a': 0, 'd':

In [53]:
contributors[0]['weeks']

[{'w': 1248566400, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1249171200, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1249776000, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1250380800, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1250985600, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1251590400, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1252195200, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1252800000, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1253404800, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1254009600, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1254614400, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1255219200, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1255824000, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1256428800, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1257033600, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1257638400, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1258243200, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1258848000, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1259452800, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1260057600, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1260662400, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1261267200, 'a': 0, 'd': 0, 'c': 0},
 {'w': 1261872000, 'a': 0, 'd': 

In [56]:
contributors[0]["total"]

21

> Get the `total` and `login` (under `author`) fields and insert them into a dataframe. Then find out who has the most number of commits.
>
> Then, write the dataframe into a table named `pandas_contributors_commits` in duckdb.

In [ ]:
contributors_with_essential_fields = [{ "login": contributor["author"]["login"], "total": contributor["total"]} for contributor in contributors]
contributors_with_essential_fields

In [60]:
pandas_contributors_commits = pd.DataFrame(contributors_with_essential_fields)
pandas_contributors_commits

,login,total
0,ryankarlos,21
1,noatamir,21
2,moink,21
3,ganevgv,22
4,minggli,22
...,...,...
95,jorisvandenbossche,1735
96,mroeschke,1951
97,wesm,3129
98,jreback,4794


In [61]:
engine = sqla.create_engine(f'duckdb:///{parent_dir}/output/unit-2-3.db')

In [62]:
try:
    pandas_contributors_commits.to_sql("pandas_contributors_commits", engine)
    print("DataFrame successfully written to the database.")
except Exception as e:
    print(f"{e}")

DataFrame successfully written to the database.


In [63]:
engine.dispose()

## GraphQL

Github offers a new way to access its data, which is the GraphQL API. You can read more about it here: https://docs.github.com/graphql

GraphQL is introspetive, which means you can query the schema of the API itself. This is very useful when you are trying to figure out what data you can access. 

In GraphQL, to access the schema you can run the following query:

```graphql
{
  __schema {
    types {
      name
      kind
      description
    }
  }
}
```

Or to get the fields of a specific type (for example `"Release"`), you can run the following query:

```graphql
{
  __type(name: "Release") {
    name
    kind
    description
    fields {
      name
      description
    }
  }
}
```

It should return the following results (in JSON):

```json
{
  "data": {
    "__type": {
      "name": "Release",
      "kind": "OBJECT",
      "description": "A release contains the content for a release.",
      "fields": [
        {
          "name": "author",
          "description": "The author of the release"
        },
        {
          "name": "createdAt",
          "description": "Identifies the date and time when the object was created."
        },
        {
          "name": "databaseId",
          "description": "Identifies the primary key from the database."
        },
        {
          "name": "description",
          "description": "The description of the release."
        },
        {
          "name": "descriptionHTML",
          "description": "The description of this release rendered to HTML."
        },
        {
          "name": "id",
          "description": null
        },
        {
          "name": "isDraft",
          "description": "Whether or not the release is a draft"
        },
        {
          "name": "isLatest",
          "description": "Whether or not the release is the latest releast"
        },
        {
          "name": "isPrerelease",
          "description": "Whether or not the release is a prerelease"
        },
        {
          "name": "mentions",
          "description": "A list of users mentioned in the release description"
        },
        {
          "name": "name",
          "description": "The title of the release."
        },
        {
          "name": "publishedAt",
          "description": "Identifies the date and time when the release was created."
        },
        {
          "name": "reactionGroups",
          "description": "A list of reactions grouped by content left on the subject."
        },
        {
          "name": "reactions",
          "description": "A list of Reactions left on the Issue."
        },
        {
          "name": "releaseAssets",
          "description": "List of releases assets which are dependent on this release."
        },
        {
          "name": "repository",
          "description": "The repository that the release belongs to."
        },
        {
          "name": "resourcePath",
          "description": "The HTTP path for this issue"
        },
        {
          "name": "shortDescriptionHTML",
          "description": "A description of the release, rendered to HTML without any links in it."
        },
        {
          "name": "tag",
          "description": "The Git tag the release points to"
        },
        {
          "name": "tagCommit",
          "description": "The tag commit for this release."
        },
        {
          "name": "tagName",
          "description": "The name of the release's Git tag"
        },
        {
          "name": "updatedAt",
          "description": "Identifies the date and time when the object was last updated."
        },
        {
          "name": "url",
          "description": "The HTTP URL for this issue"
        },
        {
          "name": "viewerCanReact",
          "description": "Can user react to this subject"
        }
      ]
    }
  }
}
```

Follow the guide [here](https://docs.github.com/en/graphql/guides/introduction-to-graphql#discovering-the-graphql-api) for more info.

In [64]:
# Feel free to test the query below - below query is to access the schema
query = """
query {
  __schema {
    types {
      name
      kind
      description
    }
  }
}
"""
            
response = requests.post("https://api.github.com/graphql", 
                        headers={"Authorization": f"Bearer {access_token}"}, json={"query": query})
response.json()

{'data': {'__schema': {'types': [{'name': 'AbortQueuedMigrationsInput',
     'kind': 'INPUT_OBJECT',
     'description': 'Autogenerated input type of AbortQueuedMigrations'},
    {'name': 'AbortQueuedMigrationsPayload',
     'kind': 'OBJECT',
     'description': 'Autogenerated return type of AbortQueuedMigrations.'},
    {'name': 'AbortRepositoryMigrationInput',
     'kind': 'INPUT_OBJECT',
     'description': 'Autogenerated input type of AbortRepositoryMigration'},
    {'name': 'AbortRepositoryMigrationPayload',
     'kind': 'OBJECT',
     'description': 'Autogenerated return type of AbortRepositoryMigration.'},
    {'name': 'AcceptEnterpriseAdministratorInvitationInput',
     'kind': 'INPUT_OBJECT',
     'description': 'Autogenerated input type of AcceptEnterpriseAdministratorInvitation'},
    {'name': 'AcceptEnterpriseAdministratorInvitationPayload',
     'kind': 'OBJECT',
     'description': 'Autogenerated return type of AcceptEnterpriseAdministratorInvitation.'},
    {'name': 'Acc

In [65]:
# Feel free to test the query below - below query is to get the fields of a specific type (for example `"Release"`)
query = """
query {
  __type(name: "Release") {
    name
    kind
    description
    fields {
      name
      description
    }
  }
}
"""
            
response = requests.post("https://api.github.com/graphql", 
                        headers={"Authorization": f"Bearer {access_token}"}, json={"query": query})
response.json()

{'data': {'__type': {'name': 'Release',
   'kind': 'OBJECT',
   'description': 'A release contains the content for a release.',
   'fields': [{'name': 'author', 'description': 'The author of the release'},
    {'name': 'createdAt',
     'description': 'Identifies the date and time when the object was created.'},
    {'name': 'databaseId',
     'description': 'Identifies the primary key from the database.'},
    {'name': 'description', 'description': 'The description of the release.'},
    {'name': 'descriptionHTML',
     'description': 'The description of this release rendered to HTML.'},
    {'name': 'id', 'description': 'The Node ID of the Release object'},
    {'name': 'immutable',
     'description': 'Whether or not the release is immutable'},
    {'name': 'isDraft',
     'description': 'Whether or not the release is a draft'},
    {'name': 'isLatest',
     'description': 'Whether or not the release is the latest releast'},
    {'name': 'isPrerelease',
     'description': 'Whether 

You can also get the schema via a `GET` request to the endpoint `https://api.github.com/graphql`:

In [66]:
response = requests.get("https://api.github.com/graphql", 
                        headers={"Authorization": f"Bearer {access_token}"})

In [67]:
schemas = response.json()

In [72]:
schemas['data']

{'__schema': {'queryType': {'name': 'Query'},
  'mutationType': {'name': 'Mutation'},
  'subscriptionType': None,
  'types': [{'kind': 'INPUT_OBJECT',
    'name': 'AbortQueuedMigrationsInput',
    'description': 'Autogenerated input type of AbortQueuedMigrations',
    'fields': None,
    'inputFields': [{'name': 'clientMutationId',
      'description': 'A unique identifier for the client performing the mutation.',
      'type': {'kind': 'SCALAR', 'name': 'String', 'ofType': None},
      'defaultValue': None},
     {'name': 'ownerId',
      'description': 'The ID of the organization that is running the migrations.',
      'type': {'kind': 'NON_NULL',
       'name': None,
       'ofType': {'kind': 'SCALAR', 'name': 'ID', 'ofType': None}},
      'defaultValue': None}],
    'interfaces': None,
    'enumValues': None,
    'possibleTypes': None},
   {'kind': 'OBJECT',
    'name': 'AbortQueuedMigrationsPayload',
    'description': 'Autogenerated return type of AbortQueuedMigrations.',
    'fi

In [74]:
schemas['data']['__schema'].keys()

dict_keys(['queryType', 'mutationType', 'subscriptionType', 'types', 'directives'])

In [75]:
len(schemas['data']['__schema']['types'])

1756

In [81]:
len(schemas['data']['__schema']['types'][0])

8

In [78]:
for t in schemas['data']['__schema']['types']:
    print(t)

{'kind': 'INPUT_OBJECT', 'name': 'AbortQueuedMigrationsInput', 'description': 'Autogenerated input type of AbortQueuedMigrations', 'fields': None, 'inputFields': [{'name': 'clientMutationId', 'description': 'A unique identifier for the client performing the mutation.', 'type': {'kind': 'SCALAR', 'name': 'String', 'ofType': None}, 'defaultValue': None}, {'name': 'ownerId', 'description': 'The ID of the organization that is running the migrations.', 'type': {'kind': 'NON_NULL', 'name': None, 'ofType': {'kind': 'SCALAR', 'name': 'ID', 'ofType': None}}, 'defaultValue': None}], 'interfaces': None, 'enumValues': None, 'possibleTypes': None}
{'kind': 'OBJECT', 'name': 'AbortQueuedMigrationsPayload', 'description': 'Autogenerated return type of AbortQueuedMigrations.', 'fields': [{'name': 'clientMutationId', 'description': 'A unique identifier for the client performing the mutation.', 'args': [], 'type': {'kind': 'SCALAR', 'name': 'String', 'ofType': None}, 'isDeprecated': False, 'deprecationR

It returns the same result as the `__schema` query above.

The GraphQL API only has a single endpoint:

```
https://api.github.com/graphql
```

In REST, HTTP verbs determine the operation performed. In GraphQL, you'll provide a JSON-encoded body whether you're performing a `query` or a `mutation`, so the HTTP verb is `POST`. The exception is an introspection query, which is a simple `GET` to the endpoint (which we did above).

To query GraphQL via `requests`, make a POST request with a JSON payload. The payload must contain a string called `query`.

GraphQL queries return only the data you specify. To form a query, you must specify fields within fields (also known as nested subfields) until you return only scalars.

For example, let's replicate the REST API exercise above where we extract the `releases`. You can see the resources in GraphQL which you can access [here](https://docs.github.com/en/graphql/reference/objects) - search for `releases`. 

We must specify the fields we want to extract:

In [85]:
query = """
query {
    repository(owner: "pandas-dev", name: "pandas") {
        releases(first: 100) {
            totalCount
            edges {
                node {
                    tagName
                    description
                    publishedAt
                }
            }
        }
    }
}
"""
            

response = requests.post("https://api.github.com/graphql", 
                        headers={"Authorization": f"Bearer {access_token}"}, json={"query": query})

Looking at the composition line by line:

`query {`

Because we want to read data from the server, not modify it, query is the root operation. (If you don't specify an operation, query is also the default.)

`repository(owner:"pandas-dev", name:"pandas") {`

To begin the query, we want to find a repository object. The schema validation indicates this object requires an owner and a name argument.

`releases(first: 100) {`

To account for all releases in the repository, we call the releases object. (We could query a single release on a repository, but that would require us to know the tagName of the release we want to return and provide it as an argument.)

Some details about the releases object:

The docs tell us this object has the type `ReleaseConnection`. Schema validation indicates this object requires a last or first number of results as an argument, so we provide first 100. You can find more information about the `ReleaseConnection` type [here](https://docs.github.com/en/graphql/reference/objects#releaseconnection).

- `totalCount`

    The beauty of GraphQL is that we can retrieve the totalCount of the releases object by simply adding it to the query and it will be returned.

- `edges {`

    We know releases is a connection because it has the ReleaseConnection type. To retrieve data about individual release, we have to access the node via edges.

- `nodes {`

    Here we retrieve the nodes at the end of the edge. The ReleaseConnection docs indicate the nodes at the end of the ReleaseConnection type is a Release object.

    Now that we know we're retrieving a Release object, we can look at the docs and specify the fields we want to return:
    ```
    tagName
    description
    publishedAt
    ```

    Here we specify the tagName, description, and publishedAt as before.

In [86]:
releases = response.json()

In [87]:
type(releases)

dict

In [88]:
releases

{'data': {'repository': {'releases': {'totalCount': 120,
    'edges': [{'node': {'tagName': 'v3.0.3',
       'description': 'We are pleased to announce the release of pandas 3.0.3.\r\nThis is a patch release in the 3.0.x series and includes some regression fixes and bug fixes. We recommend that all users of the 3.0.x series upgrade to this version.\r\n\r\nSee the [full whatsnew](https://pandas.pydata.org/docs/whatsnew/v3.0.3.html) for a list of all the changes.\r\n\r\nPandas 3.0 supports Python 3.11 and higher. \r\nThe release can be installed from PyPI:\r\n\r\n    python -m pip install --upgrade pandas==3.0.*\r\n\r\nOr from conda-forge\r\n\r\n    conda install -c conda-forge pandas=3.0\r\n\r\nPlease report any issues with the release on the [pandas issue tracker](https://github.com/pandas-dev/pandas/issues).\r\n\r\nThanks to all the contributors who made this release possible.',
       'publishedAt': '2026-05-11T18:23:29Z'}},
     {'node': {'tagName': 'v3.0.2',
       'description': '

In [89]:
releases['data']['repository']['releases']['totalCount']

120

In [ ]:
len(releases['data']['repository']['releases']['edges'])

1

In [94]:
releases['data']['repository']['releases']['edges'][0]['node']

{'tagName': 'v3.0.3',
 'description': 'We are pleased to announce the release of pandas 3.0.3.\r\nThis is a patch release in the 3.0.x series and includes some regression fixes and bug fixes. We recommend that all users of the 3.0.x series upgrade to this version.\r\n\r\nSee the [full whatsnew](https://pandas.pydata.org/docs/whatsnew/v3.0.3.html) for a list of all the changes.\r\n\r\nPandas 3.0 supports Python 3.11 and higher. \r\nThe release can be installed from PyPI:\r\n\r\n    python -m pip install --upgrade pandas==3.0.*\r\n\r\nOr from conda-forge\r\n\r\n    conda install -c conda-forge pandas=3.0\r\n\r\nPlease report any issues with the release on the [pandas issue tracker](https://github.com/pandas-dev/pandas/issues).\r\n\r\nThanks to all the contributors who made this release possible.',
 'publishedAt': '2026-05-11T18:23:29Z'}

In [95]:
releases['data']['repository']['releases']['edges'][-1]['node']

{'tagName': 'v0.18.0rc2',
 'description': '**RELEASE CANDIDATE 2**\n\nTHIS IS NOT A PRODUCTION RELEASE\n\nCompared to RC1, this includes the update to `read_sas` as well as fixes for float indexing.\n\nThis is a major release from 0.17.1 and includes a small number of API changes, several new features,\nenhancements, and performance improvements along with a large number of bug fixes. We recommend that all\nusers upgrade to this version.\n\nHighlights include:\n-  pandas >= 0.18.0 will no longer support compatibility with Python version 2.6 [GH11273](https://github.com/pydata/pandas/issues/11273) and 3.3 [GH7718](https://github.com/pydata/pandas/issues/7718)\n- Moving and expanding window functions are now methods on Series and DataFrame, similar to `.groupby`, see [here](http://pandas-docs.github.io/pandas-docs-travis/whatsnew.html#whatsnew-0180-enhancements-moments).\n- Adding support for a `RangeIndex` as a specialized form of the `Int64Index`\n  for memory savings, see [here](http:

> Query the `issues` resource, return the first 50 issues with the fields- `title`, `createdAt` and `author`, under the `author` field, return the `login` field.
>
> Refer to this [link](https://docs.github.com/en/graphql/reference/objects#issue) for more info.

In [121]:
query = """
query {
    repository(owner: "pandas-dev", name: "pandas") {
        issues(first: 100) {
            totalCount
            edges {
                node {
                        id            
                        number        
                        title         
                        body            
                        state           
                        createdAt       
                        updatedAt       
                        closedAt        
    
                        author {
                            login       
                            }
                    }
                }
            }
        }
}

"""
            

response_issues = requests.post("https://api.github.com/graphql", 
                        headers={"Authorization": f"Bearer {access_token}"}, json={"query": query})

In [123]:
 response_issues = response_issues.json()
   

In [162]:
for r in response_issues['data']['repository']['issues']['edges']:
    node = r['node']
    issue_id = node['id']
    title = node['title']
    state = node['state']
    created = node['createdAt']
    if node['author'] ==None : 
        login = 'Not available'
    else:
        login = node['author']['login']

    print(f"ID: {issue_id} | Title: {title} | State: {state} | Created: {created} | login: {login}")
    # print(f"login: {login}")

ID: MDU6SXNzdWUzMzc3MjE= | Title: Enable element-wise comparison operations in DataMatrix objects | State: CLOSED | Created: 2010-09-29T00:45:31Z | login: wesm
ID: MDU6SXNzdWUzMzc3MjY= | Title: reindex_like function | State: CLOSED | Created: 2010-09-29T00:50:13Z | login: wesm
ID: MDU6SXNzdWUzMzc3Mjg= | Title: Binary operations on int DataMatrix | State: CLOSED | Created: 2010-09-29T00:50:52Z | login: wesm
ID: MDU6SXNzdWUzMzc3MzA= | Title: Plot keyword arguments are unused in DataFrame plot() | State: CLOSED | Created: 2010-09-29T00:51:27Z | login: wesm
ID: MDU6SXNzdWUzMzc3MzY= | Title: Python 2.7 testing | State: CLOSED | Created: 2010-09-29T00:57:00Z | login: wesm
ID: MDU6SXNzdWUzMzc5OTQ= | Title: Fix argument inconsistency in pandas.stats.moments exp-weighted functions | State: CLOSED | Created: 2010-09-29T05:30:56Z | login: wesm
ID: MDU6SXNzdWUzMzg5MDk= | Title: DataMatrix constructor ignores dtype argument if data is a DataMatrix | State: CLOSED | Created: 2010-09-29T15:41:55Z | l

## Web Scraping

In [10]:
import requests
from bs4 import BeautifulSoup

In [164]:
r = requests.get("https://www.scrapethissite.com/pages/simple/")

In [165]:
r.status_code

200

In [166]:
type(r.text)

str

In [167]:
print(r.text)

<!doctype html>
<html lang="en">
  <head>
    <meta charset="utf-8">
    <title>Countries of the World: A Simple Example | Scrape This Site | A public sandbox for learning web scraping</title>
    <link rel="icon" type="image/png" href="/static/images/scraper-icon.png" />

    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="description" content="A single page that lists information about all the countries in the world. Good for those just get started with web scraping.">

    <link href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" rel="stylesheet" integrity="sha256-MfvZlkHCEqatNoGiOXveE8FIwMzZg4W85qfrfIFBfYc= sha512-dTfge/zgoMYpP7QbHy4gWMEGsbsdZeCXz7irItjcC3sPUFtf0kuFbDz/ixG7ArTxmDjLXDmezHubeNikyKGVyQ==" crossorigin="anonymous">
    <link href='https://fonts.googleapis.com/css?family=Lato:400,700' rel='stylesheet' type='text/css'>
    <link rel="stylesheet" type="text/css" href="/static/css/styles.css">

    
<meta name=

We will use BeautifulSoup to parse the HTML above. There are many parsers available. The default parser is `html.parser`, it does not require any additional installation. However, its performance is not the best. If you want better performance, you can use `lxml` parser. Refer to [here](https://beautiful-soup-4.readthedocs.io/en/latest/index.html?highlight=get#installing-a-parser) for more info.

In [168]:
soup = BeautifulSoup(r.text, "html.parser")

We can look for all paragraphs in the page using the `find_all` method. This returns a list of all the paragraph `Tag`s in the page.

Refer to the [documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#bs4.Tag) for more details.

In [170]:
paragraphs = soup.find_all('p')

In [171]:
len(paragraphs)

3

In [172]:
type(paragraphs[0])

bs4.element.Tag

In [173]:
paragraphs[0]

<p class="lead">
                            A single page that lists information about all the countries in the world. Good for those just get started with web scraping.
                            Practice looking for patterns in the HTML that will allow you to extract information about each country. Then, build a simple web scraper that makes a request to this page, parses the HTML and prints out each country's name.
                        </p>

In [174]:
# get the actual text
paragraphs[0].text

"\n                            A single page that lists information about all the countries in the world. Good for those just get started with web scraping.\n                            Practice looking for patterns in the HTML that will allow you to extract information about each country. Then, build a simple web scraper that makes a request to this page, parses the HTML and prints out each country's name.\n                        "

We can find all the country names in the HTML by specifying the tag and class name. It is always good to be as specific as possible when selecting elements. This is because the more specific you are, the less likely you will select other unintended elements.

In [175]:
country_name_elements = soup.find_all('h3', 'country-name')

or equivalently

`soup.find_all('h3', class_='country-name')`

In [176]:
country_name_elements

[<h3 class="country-name">
 <i class="flag-icon flag-icon-ad"></i>
                             Andorra
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-ae"></i>
                             United Arab Emirates
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-af"></i>
                             Afghanistan
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-ag"></i>
                             Antigua and Barbuda
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-ai"></i>
                             Anguilla
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-al"></i>
                             Albania
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-am"></i>
                             Armenia
                         </h3>,
 <h3 class="countr

> Extract just the country names string into a new list called `country_names`. Remove all whitespaces.
>
> Your results should be starting with `['Andorra', 'United Arab Emirates', 'Afghanistan', 'Antigua and Barbuda', ...]`

In [184]:
country_name_elements = soup.find_all('h3', 'country-name')

In [185]:
country_name_elements

[<h3 class="country-name">
 <i class="flag-icon flag-icon-ad"></i>
                             Andorra
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-ae"></i>
                             United Arab Emirates
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-af"></i>
                             Afghanistan
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-ag"></i>
                             Antigua and Barbuda
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-ai"></i>
                             Anguilla
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-al"></i>
                             Albania
                         </h3>,
 <h3 class="country-name">
 <i class="flag-icon flag-icon-am"></i>
                             Armenia
                         </h3>,
 <h3 class="countr

In [188]:


country_names = []

# Loop through each h3 element individually
for element in country_name_elements:
    # Find the <i> tag inside this specific <h3>
   
        # Extract the clean text inside the <i> tag
        country_names.append(element.text.strip())
        

print(country_names)

['Andorra', 'United Arab Emirates', 'Afghanistan', 'Antigua and Barbuda', 'Anguilla', 'Albania', 'Armenia', 'Angola', 'Antarctica', 'Argentina', 'American Samoa', 'Austria', 'Australia', 'Aruba', 'Åland', 'Azerbaijan', 'Bosnia and Herzegovina', 'Barbados', 'Bangladesh', 'Belgium', 'Burkina Faso', 'Bulgaria', 'Bahrain', 'Burundi', 'Benin', 'Saint Barthélemy', 'Bermuda', 'Brunei', 'Bolivia', 'Bonaire', 'Brazil', 'Bahamas', 'Bhutan', 'Bouvet Island', 'Botswana', 'Belarus', 'Belize', 'Canada', 'Cocos [Keeling] Islands', 'Democratic Republic of the Congo', 'Central African Republic', 'Republic of the Congo', 'Switzerland', 'Ivory Coast', 'Cook Islands', 'Chile', 'Cameroon', 'China', 'Colombia', 'Costa Rica', 'Cuba', 'Cape Verde', 'Curacao', 'Christmas Island', 'Cyprus', 'Czech Republic', 'Germany', 'Djibouti', 'Denmark', 'Dominica', 'Dominican Republic', 'Algeria', 'Ecuador', 'Estonia', 'Egypt', 'Western Sahara', 'Eritrea', 'Spain', 'Ethiopia', 'Finland', 'Fiji', 'Falkland Islands', 'Micron

> Extract the capitals into a new list called `capital_names`.

It is common to collect all the required information at one go. For each country element, you can further use the `find` method to extract the child element.

Let's collect the capital, population and area for each country.

In [189]:
countries = []
for country in soup.find_all('div', 'col-md-4 country'):
    country_info = {}
    country_info['name'] = country.find('h3').text.strip()
    country_info['capital'] = country.find('span', 'country-capital').text
    country_info['population'] = country.find('span', 'country-population').text
    country_info['area_km2'] = country.find('span', 'country-area').text
    countries.append(country_info)

In [190]:
countries

[{'name': 'Andorra',
  'capital': 'Andorra la Vella',
  'population': '84000',
  'area_km2': '468.0'},
 {'name': 'United Arab Emirates',
  'capital': 'Abu Dhabi',
  'population': '4975593',
  'area_km2': '82880.0'},
 {'name': 'Afghanistan',
  'capital': 'Kabul',
  'population': '29121286',
  'area_km2': '647500.0'},
 {'name': 'Antigua and Barbuda',
  'capital': "St. John's",
  'population': '86754',
  'area_km2': '443.0'},
 {'name': 'Anguilla',
  'capital': 'The Valley',
  'population': '13254',
  'area_km2': '102.0'},
 {'name': 'Albania',
  'capital': 'Tirana',
  'population': '2986952',
  'area_km2': '28748.0'},
 {'name': 'Armenia',
  'capital': 'Yerevan',
  'population': '2968000',
  'area_km2': '29800.0'},
 {'name': 'Angola',
  'capital': 'Luanda',
  'population': '13068161',
  'area_km2': '1246700.0'},
 {'name': 'Antarctica',
  'capital': 'None',
  'population': '0',
  'area_km2': '1.4E7'},
 {'name': 'Argentina',
  'capital': 'Buenos Aires',
  'population': '41343201',
  'area_km2

Let's scrape a more complex example. This page has a table with pagination. We will scrape the rows in all the pages and store the data in a dataframe.

[https://www.scrapethissite.com/pages/forms/](https://www.scrapethissite.com/pages/forms/)

First, as with any web scraping task, we need to inspect the page to understand the structure of the page.

We'll start by scraping the first page:

In [191]:
r = requests.get("https://www.scrapethissite.com/pages/forms/")

In [192]:
r.status_code

200

In [193]:
soup = BeautifulSoup(r.text, "html.parser")

Let's extract the table header first:

In [198]:
header = soup.find('tr')

In [199]:
header

<tr>
<th>
                            Team Name
                        </th>
<th>
                            Year
                        </th>
<th>
                            Wins
                        </th>
<th>
                            Losses
                        </th>
<th>
                            OT Losses
                        </th>
<th>
                            Win %
                        </th>
<th>
                            Goals For (GF)
                        </th>
<th>
                            Goals Against (GA)
                        </th>
<th>
                            + / -
                        </th>
</tr>

In [202]:
headers = [th.text.strip() for th in header.find_all('th')]

In [203]:
headers

['Team Name',
 'Year',
 'Wins',
 'Losses',
 'OT Losses',
 'Win %',
 'Goals For (GF)',
 'Goals Against (GA)',
 '+ / -']

Let's find all the rows on first page:

In [204]:
teams = soup.find_all('tr', 'team')

In [206]:
teams[0]

<tr class="team">
<td class="name">
                            Boston Bruins
                        </td>
<td class="year">
                            1990
                        </td>
<td class="wins">
                            44
                        </td>
<td class="losses">
                            24
                        </td>
<td class="ot-losses">
</td>
<td class="pct text-success">
                            0.55
                        </td>
<td class="gf">
                            299
                        </td>
<td class="ga">
                            264
                        </td>
<td class="diff text-success">
                            35
                        </td>
</tr>

In [207]:
for col in teams[0].find_all('td'):
    print(col.text.strip())

Boston Bruins
1990
44
24

0.55
299
264
35


We can combine that with the header:

In [210]:
row_dict = {}
for header, col in zip(headers, teams[0].find_all('td')):
    row_dict[header] = col.text.strip()

In [212]:
row_dict

{'Team Name': 'Boston Bruins',
 'Year': '1990',
 'Wins': '44',
 'Losses': '24',
 'OT Losses': '',
 'Win %': '0.55',
 'Goals For (GF)': '299',
 'Goals Against (GA)': '264',
 '+ / -': '35'}

> Repeat the operation above for all the teams (rows) using a `for` loop, and append each row_dict to a list called `rows`.

In [216]:
teams

[<tr class="team">
 <td class="name">
                             Boston Bruins
                         </td>
 <td class="year">
                             1990
                         </td>
 <td class="wins">
                             44
                         </td>
 <td class="losses">
                             24
                         </td>
 <td class="ot-losses">
 </td>
 <td class="pct text-success">
                             0.55
                         </td>
 <td class="gf">
                             299
                         </td>
 <td class="ga">
                             264
                         </td>
 <td class="diff text-success">
                             35
                         </td>
 </tr>,
 <tr class="team">
 <td class="name">
                             Buffalo Sabres
                         </td>
 <td class="year">
                             1990
                         </td>
 <td class="wins">
                             3

In [217]:
rows_dict = {}
list_rows = []
for team in teams:
    for header, col in zip(headers, team.find_all('td')):
        rows_dict[header] = col.text.strip()
        list_rows.append(rows_dict)

In [218]:
list_rows

[{'Team Name': 'Chicago Blackhawks',
  'Year': '1991',
  'Wins': '36',
  'Losses': '29',
  'OT Losses': '',
  'Win %': '0.45',
  'Goals For (GF)': '257',
  'Goals Against (GA)': '236',
  '+ / -': '21'},
 {'Team Name': 'Chicago Blackhawks',
  'Year': '1991',
  'Wins': '36',
  'Losses': '29',
  'OT Losses': '',
  'Win %': '0.45',
  'Goals For (GF)': '257',
  'Goals Against (GA)': '236',
  '+ / -': '21'},
 {'Team Name': 'Chicago Blackhawks',
  'Year': '1991',
  'Wins': '36',
  'Losses': '29',
  'OT Losses': '',
  'Win %': '0.45',
  'Goals For (GF)': '257',
  'Goals Against (GA)': '236',
  '+ / -': '21'},
 {'Team Name': 'Chicago Blackhawks',
  'Year': '1991',
  'Wins': '36',
  'Losses': '29',
  'OT Losses': '',
  'Win %': '0.45',
  'Goals For (GF)': '257',
  'Goals Against (GA)': '236',
  '+ / -': '21'},
 {'Team Name': 'Chicago Blackhawks',
  'Year': '1991',
  'Wins': '36',
  'Losses': '29',
  'OT Losses': '',
  'Win %': '0.45',
  'Goals For (GF)': '257',
  'Goals Against (GA)': '236',
  '

In [215]:
rows_dict

{'Team Name': 'Chicago Blackhawks',
 'Year': '1991',
 'Wins': '36',
 'Losses': '29',
 'OT Losses': '',
 'Win %': '0.45',
 'Goals For (GF)': '257',
 'Goals Against (GA)': '236',
 '+ / -': '21'}

If you press on any page number (button) in the bottom, you will see the URL change, for example pressing on page 2:

`https://www.scrapethissite.com/pages/forms/?page_num=2`

This is because the page number is a parameter in the URL. We can use this to our advantage to scrape all the pages. We can use a `for loop` to iterate through all the pages and scrape the data from each page.

In [11]:

def parse_and_extract_rows(response_text: str):
    """
    Parse the HTML and extract table rows from the response object's text.
    
    Args:
        response_text (str): The HTML text from the response object.
        
    Returns:
        An iterator of dictionaries with the data from the current page.
    """
    soup = BeautifulSoup(response_text, "html.parser")
    header = soup.find('tr')
    headers = [th.text.strip() for th in header.find_all('th')]
    teams = soup.find_all('tr', 'team')
    for team in teams:
        row_dict = {}
        for header, col in zip(headers, team.find_all('td')):
            row_dict[header] = col.text.strip()
        yield row_dict

As there are 24 pages, we need to iterate through all of them to get all the data we want. We will use a for loop to do this. We will also use the `time.sleep()` function to pause and make sure we don't overload the server with requests.

In [12]:
import time

In [14]:
rows = []
for page in range(1, 2):
    r = requests.get(f"https://www.scrapethissite.com/pages/forms/?page_num={page}")
    for row_dict in parse_and_extract_rows(r.text):
        rows.append(row_dict)
    # pause for 1 second between requests
    time.sleep(1)

In [222]:
len(rows)

582

In [223]:
rows[0]

{'Team Name': 'Boston Bruins',
 'Year': '1990',
 'Wins': '44',
 'Losses': '24',
 'OT Losses': '',
 'Win %': '0.55',
 'Goals For (GF)': '299',
 'Goals Against (GA)': '264',
 '+ / -': '35'}

In [224]:
rows[-1]

{'Team Name': 'Winnipeg Jets',
 'Year': '2011',
 'Wins': '37',
 'Losses': '35',
 'OT Losses': '10',
 'Win %': '0.451',
 'Goals For (GF)': '225',
 'Goals Against (GA)': '246',
 '+ / -': '-21'}

Let's convert the list of dictionaries into a dataframe to perform further analysis.

In [225]:
df = pd.DataFrame(rows)

 Cast the correct dtypes for the columns, we'll use pandas [nullable](https://pandas.pydata.org/pandas-docs/stable/user_guide/basics.html#basics-dtypes) integer (`pd.Int64Dtype`) and float (`pd.Float64Dtype`) type for the numeric columns.

 Below, `int` is an alias for `np.int64`, while `Int64` represents `pd.Int64Dtype`.

In [226]:
df = df.replace('', pd.NA).astype({'Year': int, 'Wins': 'Int64', 'Losses': 'Int64', 'OT Losses': 'Int64', 
                                   'Goals For (GF)': 'Int64', 'Goals Against (GA)': 'Int64', 
                                   '+ / -': 'Int64', 'Win %': float})

In [259]:
df.sort_values(by = 'Wins',ascending=False)

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
126,Detroit Red Wings,1995,62,13,<NA>,0.756,325,181,144
382,Detroit Red Wings,2005,58,16,8,0.707,305,209,96
58,Pittsburgh Penguins,1992,56,21,<NA>,0.667,367,268,99
550,Vancouver Canucks,2010,54,19,9,0.659,262,185,77
521,Washington Capitals,2009,54,15,13,0.659,318,233,85
...,...,...,...,...,...,...,...,...,...
82,Ottawa Senators,1993,14,61,<NA>,0.167,201,397,-196
225,Atlanta Thrashers,1999,14,57,4,0.171,170,313,-143
60,San Jose Sharks,1992,11,71,<NA>,0.131,218,414,-196
56,Ottawa Senators,1992,10,70,<NA>,0.119,202,395,-193


> 1. Find the team that has the most number of Wins for the year.
> 
> 2. Find the team that has the most number of Losses for the year.
> 
> 3. Find the team that has the lowest Win % for the year.
> 
> 4. Find the team that has the most number of Wins for all years.

In [270]:
# 1. Filter data for the specific target year (e.g., 2026)
target_year = 2026
df_year = df[df['Year'] == target_year].copy()

# # 2. Add Win % column for the third question
# df_year['Win_Pct'] = df_year['Wins'] / (df_year['Wins'] + df_year['Losses'])

# 3. Aggregate all-time wins for the fourth question
all_time_wins = df.groupby('Team Name')['Wins'].sum()
wins_by_year = df.groupby(['Team Name','Year'])['Wins'].sum()
losses_by_year = df.groupby(['Team Name','Year'])['Losses'].sum()
# print(all_time_wins.sort_values(ascending=False).head(1))

print(all_time_wins.idxmax())
print(wins_by_year.sort_values(ascending=False).head(5))
print(losses_by_year.idxmax)

# 4. Compile all answers into a dictionary
# results = {
#     f"Most Wins ({target_year})": df.loc[df_year['Wins'].idxmax(), 'Team Name'],
#     f"Most Losses ({target_year})": df.loc[df_year['Losses'].idxmax(), 'Team Name'],
#     f"Lowest Win % ({target_year})": df.loc[df_year['Win_Pct'].idxmin(), 'Team Name'],
#     "Most Wins (All Years)": all_time_wins.idxmax()
# }

# # 5. Print the results clearly
# for question, team in results.items():
#     print(f"{question}: {team}")


Detroit Red Wings
Team Name            Year
Detroit Red Wings    1995    62
                     2005    58
Pittsburgh Penguins  1992    56
Detroit Red Wings    2007    54
Vancouver Canucks    2010    54
Name: Wins, dtype: Int64
<bound method Series.idxmax of Team Name      Year
Anaheim Ducks  2006    20
               2007    27
               2008    33
               2009    32
               2010    30
                       ..
Winnipeg Jets  1992    37
               1993    51
               1994    25
               1995    40
               2011    35
Name: Losses, Length: 582, dtype: Int64>


In [ ]:
top_team = df.loc[df['Losses'].idxmax(), 'Team Name']
top_team

In [234]:
df.dtypes

Team Name              object
Year                    int64
Wins                    Int64
Losses                  Int64
OT Losses               Int64
Win %                 float64
Goals For (GF)          Int64
Goals Against (GA)      Int64
+ / -                   Int64
dtype: object

In [239]:
df.groupby('Team Name').agg({'Wins': 'max', 'Losses': 'max' , 'Win %':'min'})

,Wins,Losses,Win %
Team Name,,,
Anaheim Ducks,48,36,0.415
Atlanta Thrashers,43,57,0.171
Boston Bruins,53,47,0.293
Buffalo Sabres,53,42,0.329
Calgary Flames,46,41,0.317
Carolina Hurricanes,52,43,0.268
Chicago Blackhawks,52,43,0.244
Colorado Avalanche,52,45,0.366
Columbus Blue Jackets,41,47,0.268
